In [7]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q accelerate
!pip install -q pypdf
!pip install -q fastapi uvicorn nest_asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.9 MB/s eta 0:00:00


In [18]:
!pip install -q langchain_text_splitters langchain_huggingface
!pip install -q --upgrade transformers

from google.colab import files
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline

# -------------------------
# Upload PDF
# -------------------------

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

# -------------------------
# Read PDF
# -------------------------

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("="*70)
print("PDF Loaded Successfully")
print("="*70)

# -------------------------
# Split Text
# -------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.create_documents([text])

# -------------------------
# Create FAISS Database
# -------------------------

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(docs, embedding)

print("FAISS Vector Database Created")
print()

# -------------------------
# Load AI Models
# -------------------------

summarizer = pipeline(
    "text-generation",
    model="facebook/bart-large-cnn"
)

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

# -------------------------
# Summary
# -------------------------

summary = summarizer(
    text[:2000],
    max_length=150,
    min_length=50,
    do_sample=False
)[0]["generated_text"]

print("="*70)
print("SUMMARY")
print("="*70)
print(summary)

# -------------------------
# Revision Notes
# -------------------------

prompt = f"""
Create short revision notes.

{text[:1500]}
"""

revision = generator(
    prompt,
    max_length=200
)[0]["generated_text"]

print("\n")
print("="*70)
print("REVISION NOTES")
print("="*70)
print(revision)

# -------------------------
# Quiz Questions
# -------------------------

quiz_prompt = f"""
Generate five quiz questions.

{text[:1500]}
"""

quiz = generator(
    quiz_prompt,
    max_length=250
)[0]["generated_text"]

print("\n")
print("="*70)
print("QUIZ QUESTIONS")
print("="*70)
print(quiz)

# -------------------------
# Question Answering
# -------------------------

question = "Explain the main topic."

docs = db.similarity_search(question, k=2)

context = ""

for d in docs:
    context += d.page_content

qa_prompt = f"""
Context:

{context}

Question:
{question}

Answer:
"""

answer = generator(
    qa_prompt,
    max_length=200
)[0]["generated_text"]

print("\n")
print("="*70)
print("QUESTION ANSWERING")
print("="*70)
print("Question:", question)
print("Answer:", answer)

Saving _OceanofPDF.com_The_Parted_Earth_-_Anjali_Enjeti.pdf to _OceanofPDF.com_The_Parted_Earth_-_Anjali_Enjeti (1).pdf
PDF Loaded Successfully


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS Vector Database Created



Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.bias       | UNEXPECTED |  | 
mod

...



REVISION NOTES

Create short revision notes.

THE PARTED EARTH
OceanofPDF.comOceanofPDF.comCopyright © 2021
Anjali Enjeti
All rights reserved. No part of this book may be reproduced in any form or by any electronic means,
including information storage and retrieval systems, without permission in writing from the
publisher, except by a reviewer, who may quote brief passages in a review.
Design Lead: Meg Reid
Proofreaders: Kalee Lineberger, Kendall Owens, Stephanie Trott
Library of Congress Cataloging-in-Publication Data
Names: Enjeti, Anjali, author.
Title: The parted earth / Anjali Enjeti.
Description: Spartanburg, SC : Hub City Press, [2021]
Summary: “Spanning more than half a century and cities from New Delhi to Atlanta, Anjali Enjeti’s
debut is a heartfelt and human portrait of the long shadow of the Partition of the Indian subcontinent
on the lives of three generations.”
Identifiers:
LCCN 2020046269
ISBN 9781938235771 (hardcover)
ISBN 9781938235788 (ebook)
Subjects: LCSH: India--

[transformers] Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




QUIZ QUESTIONS

Generate five quiz questions.

THE PARTED EARTH
OceanofPDF.comOceanofPDF.comCopyright © 2021
Anjali Enjeti
All rights reserved. No part of this book may be reproduced in any form or by any electronic means,
including information storage and retrieval systems, without permission in writing from the
publisher, except by a reviewer, who may quote brief passages in a review.
Design Lead: Meg Reid
Proofreaders: Kalee Lineberger, Kendall Owens, Stephanie Trott
Library of Congress Cataloging-in-Publication Data
Names: Enjeti, Anjali, author.
Title: The parted earth / Anjali Enjeti.
Description: Spartanburg, SC : Hub City Press, [2021]
Summary: “Spanning more than half a century and cities from New Delhi to Atlanta, Anjali Enjeti’s
debut is a heartfelt and human portrait of the long shadow of the Partition of the Indian subcontinent
on the lives of three generations.”
Identifiers:
LCCN 2020046269
ISBN 9781938235771 (hardcover)
ISBN 9781938235788 (ebook)
Subjects: LCSH: India-

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




QUESTION ANSWERING
Question: Explain the main topic.
Answer: 
Context:

Chapter Eight
Chapter Nine
Chapter Ten
Chapter Eleven
Part Three
Chapter Twelve
Chapter Thirteen
Chapter Fourteen
Chapter FifteenChapter Sixteen
Chapter Seventeen
Chapter Eighteen
Chapter Nineteen
Chapter Twenty
Chapter Twenty-One
Chapter Twenty-Two
Chapter Twenty-Three
Chapter Twenty-Four
Chapter Twenty-Five
Chapter Twenty-Six
Chapter Twenty-Seven
Acknowledgements
Further Reading About Partition
OceanofPDF.comT
AUTHOR’S NOTEthe moon, the earth to the sun, the order of the planets, their sizes, from
largest to smallest, how the Vedic verses, thousands of years old, first
introduced the concept that the sun was the center of the universe, the earth
was the shape of a sphere.
He never met anyone as smart as his sister.
He tilted his head. The stars were specks like tiny particles of dust,
flames just catching on a wick. It comforted him to know that the sky

Question:
Explain the main topic.

Answer:

